In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from edc_pdutils.dataframes import get_subject_visit
from edc_pdutils.dataframes import get_crf
from intecomm_analytics.dataframes import get_df_main_1858
from django.apps import apps as django_apps
from bs4 import BeautifulSoup
from intecomm_analytics.dataframes.main_1858_to_stata import to_stata
import re

In [ ]:
def strip_html(text:str)->str:
    if pd.isna(text):
        return text
    if bool(re.search(r'<[^>]+>', text)):
        return BeautifulSoup(text, "html.parser").get_text()
    return text

def get_model_and_merge_with_df_main(df:pd.DataFrame, model:str, suffix:str)->pd.DataFrame:
    _, model_name = model.split(".")
    df_crf = get_crf(model, subject_visit_model="intecomm_subject.subjectvisit", read_verbose=False)
    df_crf = (
        df_crf[[col for col in df_crf.columns if col not in visit_columns and col not in system_columns]]
        .copy()
        .rename(columns={col:f"{col}_{suffix}" for col in df_crf.columns if col not in ["subject_visit_id", ]})
    )
    df_crf[f"crf_{model_name}"] = 1
    df = df.merge(df_crf, on="subject_visit_id", how="left", suffixes=("", f"_{suffix}"))
    return df


def get_stata_labels(df:pd.DataFrame, model:str)->dict[str:str]:
    """Generate STATA labels"""
    labels = {}
    _, model_name = model.split(".")
    model_cls = django_apps.get_model(model)
    for fld in model_cls._meta.get_fields():
        if f"{fld.name}_{model_name}" in df.columns:
            labels.update({f"{fld.name}_{model_name}": strip_html(str(fld.verbose_name)[:80])})
    return labels

In [ ]:
variable_labels = {}

df_main_1868 = get_df_main_1858(None, fasting_hours=8.0)

df_visit = get_subject_visit("intecomm_subject.subjectvisit")
system_columns = ["id", "consent_model", "consent_version", "crf_status", "crf_status_comments", "created", "modified", "user_created", "user_modified", "hostname_created", "hostname_modified", "device_created", "device_modified", "locale_created", "locale_modified", "revision"]
visit_columns = [col for col in df_visit.columns if col != "subject_visit_id"]

df_main = df_visit.copy()
df_main = df_main.merge(
    df_main_1868[[
        "subject_identifier",
        "country",
        "assignment",
        "group_identifier",
        "hiv",
        "dm",
        "htn",
        "age_in_years",
        "gender", *[col for col in df_main_1868.columns if col.startswith('primary')]
    ]], how="left", on="subject_identifier")

In [ ]:
# eq5d3l
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.eq5d3l", "eq5d3l")
variable_labels.update(get_stata_labels(df_main, "intecomm_subject.eq5d3l"))

In [ ]:
# icecapa
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.icecapa", "icecapa")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.icecapa"))

In [ ]:
# assets
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsassets", "assets")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.healtheconomicsassets"))

In [ ]:
# csa
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.careseekinga", "careseekinga")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.careseekinga"))

In [ ]:
# csb
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.careseekingb", "careseekingb")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.careseekingb"))

In [ ]:
# hh
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicshouseholdhead", "householdhead")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.healtheconomicshouseholdhead"))

In [ ]:
# income
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsincome", "income")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.healtheconomicsincome"))

In [ ]:
# patient
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicspatient", "patient")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.healtheconomicspatient"))

In [ ]:
# property
df_main = get_model_and_merge_with_df_main(df_main, "intecomm_subject.healtheconomicsproperty", "property")
variable_labels.update(get_stata_labels(df_main,"intecomm_subject.healtheconomicsproperty"))

In [ ]:
df_main


In [ ]:
# these are the long field names
# go through this by hand and shorten
original_labels = ['primary_vl_controlled_baseline_400', 'primary_vl_controlled_baseline_50', 'primary_vl_controlled_endline_400', 'health_today_score_confirmed_eq5d3l', 'external_wall_material_other_assets', 'external_window_material_other_assets', 'care_visit_reason_other_careseekinga', 'med_conditions_other_careseekinga', 'med_not_collected_reason_careseekinga', 'med_not_collected_reason_other_careseekinga', 'med_collected_location_careseekinga', 'med_collected_location_other_careseekinga', 'tests_not_done_reason_careseekinga', 'tests_not_done_other_careseekinga', 'missed_activities_other_careseekinga', 'no_accessed_care_other_careseekingb', 'med_conditions_other_careseekingb', 'med_not_collected_reason_careseekingb', 'med_not_collected_reason_other_careseekingb', 'tests_not_done_reason_careseekingb', 'tests_not_done_other_careseekingb', 'missed_activities_other_careseekingb', 'inpatient_reasons_other_careseekingb', 'inpatient_nowork_days_careseekingb', 'inpatient_household_nowork_careseekingb', 'inpatient_household_nowork_days_careseekingb', 'inpatient_money_sources_other_careseekingb', 'inpatient_money_sources_main_careseekingb', 'inpatient_money_sources_main_other_careseekingb', 'relationship_to_hoh_householdhead', 'relationship_to_hoh_other_householdhead', 'hoh_ethnicity_other_householdhead', 'hoh_education_other_householdhead', 'hoh_employment_status_householdhead', 'hoh_employment_type_householdhead', 'hoh_employment_type_other_householdhead', 'hoh_marital_status_other_householdhead', 'hoh_insurance_other_householdhead', 'hoh_employment_type_old_householdhead', 'ngo_assistance_value_known_income', 'internal_remit_value_known_income', 'external_remit_value_known_income', 'external_remit_currency_other_income', 'pat_employment_type_other_patient', 'calculated_land_surface_area_property']
shortened_labels = ['primary_vl_cntrl_baseline_400', 'primary_vl_cntrl_baseline_50', 'primary_vl_cntrl_endline_400', 'health_today_score_conf_eq5d3l', 'ext_wall_material_other_assets', 'ext_window_material_other_assets', 'care_visit_reason_other_csa', 'med_cond_other_csa', 'med_not_coll_reason_csa', 'med_not_coll_reason_other_csa', 'med_coll_location_csa', 'med_coll_location_other_csa', 'tests_not_done_reason_csa', 'tests_not_done_other_csa', 'missed_activities_other_csa', 'no_accessed_care_other_csb', 'med_cond_other_csb', 'med_not_coll_reason_csb', 'med_not_coll_reason_other_csb', 'tests_not_done_reason_csb', 'tests_not_done_other_csb', 'missed_activities_other_csb', 'inpat_reasons_other_csb', 'inpat_nowork_days_csb', 'inpat_hh_nowork_csb', 'inpat_hh_nowork_days_csb', 'inpat_money_src_other_csb', 'inpat_money_src_main_csb', 'inpat_money_src_main_other_csb', 'rela_to_hoh_hhh', 'rela_to_hoh_other_hhh', 'hoh_ethnicity_other_hhh', 'hoh_education_other_hhh', 'hoh_emply_status_hhh', 'hoh_emply_type_hhh', 'hoh_emply_type_other_hhh', 'hoh_marital_status_other_hhh', 'hoh_insurance_other_hhh', 'hoh_emply_type_old_hhh', 'ngo_asst_value_known_income', 'int_remit_value_known_income', 'ext_remit_value_known_income', 'ext_remit_curr_other_income', 'pat_emply_type_other_patient', 'calc_land_surf_area_property']

In [ ]:
# rename cols in dataframe using shortened col names
rename_cols = dict(zip(original_labels, shortened_labels))
df_main = df_main.rename(columns=rename_cols)

In [ ]:
# update the variable labels for stata wih the shortened col names
rev_variable_labels = {description: fld for fld, description in variable_labels.items()}
for orig_fld, shortened_fld in rename_cols.items():
    if orig_fld in variable_labels:
        rev_variable_labels[variable_labels[orig_fld]] = shortened_fld
variable_labels = {v:k for k,v in rev_variable_labels.items()}

In [ ]:
df_main.select_dtypes(include="datetime")
df_main.select_dtypes(include="int").dtypes

In [ ]:
# export
df_main["roof_material_other_assets"] = df_main["roof_material_other_assets"].fillna("")
df_main["tests_not_done_other_csa"] = df_main["tests_not_done_other_csa"].fillna("")
df_main["no_accessed_care_other_csb"] = df_main["no_accessed_care_other_csb"].fillna("")
df_main["hoh_education_other_hhh"] = df_main["hoh_education_other_hhh"].fillna("")
df_main["land_surface_area_property"] = df_main["land_surface_area_property"].astype("Float64")
df_main["calc_land_surf_area_property"] = df_main["calc_land_surf_area_property"].astype("Float64")

to_stata(df_main, analysis_folder, filename="df_he.dta", stata_labels=variable_labels)